# RoofTop FRA Scoping

## Purpose

Scope the FRA rooftop relocation set for the production run. The scope is signed off
(GO from MapOps and the Local Expert) and split into two CSV sets on blob:

| Scope_for | Count | Blob path |
|---|---|---|
| Need_to_relocate | 15,503,437 | `dbfs:/mnt/adr-map-expert/APT/FRA_RoofTop_relocation/LIVE_RUN_CSV_20260821/Need_to_relocate/` |
| Already_On_BFT | 2,828,067 | `dbfs:/mnt/adr-map-expert/APT/FRA_RoofTop_relocation/LIVE_RUN_CSV_20260821/Already_On_BFT/` |
| **Grand Total** | **18,331,504** | |

Sources: **BAN** (French national address register — address positions, street/house-number
attributes) and **RBN** (French building register — building footprints and their registered
addresses). Expected rooftop accuracy (correct building): **24.7 pp → ~85 pp**.

Reference: [SEACO-6712](https://tomtom.atlassian.net/browse/SEACO-6712)

In [0]:
display(dbutils.fs.ls("dbfs:/mnt/wth/oz/data-correctness/correction-input/SEACO-6712-FRA-Relocation/FRA_RoofTop_relocation/LIVE_RUN_CSV_20260821/Need_to_relocate/"))

In [0]:
NEED_TO_RELOCATE_PATH = "dbfs:/mnt/wth/oz/data-correctness/correction-input/SEACO-6712-FRA-Relocation/FRA_RoofTop_relocation/LIVE_RUN_CSV_20260821/Need_to_relocate/part-00000-tid-1416591313286528631-ad2f1d89-d746-4078-b440-67536c08252c-1685-1-c000.csv"

need_to_relocate_df = spark.read.csv(
    NEED_TO_RELOCATE_PATH,
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
)

need_to_relocate_df.limit(5).display()

In [ ]:
from pyspark.sql.functions import col, format_string, lit, round as spark_round

FRA_SCOPE_SPLIT_PATH = "dbfs:/mnt/wth/oz/data-correctness/correction-input/SEACO-6712-FRA-Relocation/FRA_RoofTop_relocation/FRA-Scope-Split/"

# Orbis stores coordinates as integers at 1e-7 degree precision, so the CSV must
# carry exactly 7 decimals — no truncation, no scientific notation.
DEFAULT_SCALE = 1.0e7


def to_orbis_coordinate(column_name):
    """Snap to the Orbis 1e-7 grid and render as a fixed 7-decimal string."""
    snapped = spark_round(col(column_name) * lit(DEFAULT_SCALE)) / lit(DEFAULT_SCALE)
    return format_string("%.7f", snapped)


# Correction-input template columns, mapped from the scope CSV.
# Note: Orbis_X takes Orbis_Y_New and Orbis_Y takes Orbis_X_New (lat/lng order is swapped on purpose).
scope_split_df = need_to_relocate_df.limit(5).select(
    col("Product_orbis_id"),
    col("old_location_xy").alias("Target_Value"),
    lit("metadata:apa:previous_location").alias("Product_Address_component"),
    to_orbis_coordinate("Orbis_Y_New").alias("Orbis_X"),
    to_orbis_coordinate("Orbis_X_New").alias("Orbis_Y"),
    col("Relocation_Type"),
)

(
    scope_split_df.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(FRA_SCOPE_SPLIT_PATH)
)

In [ ]:
print(f"Records written: {scope_split_df.count()} -> {FRA_SCOPE_SPLIT_PATH}")
scope_split_df.display()